## Unit Testing Flixtube

This notebook covers **Unit Testing** with [xUnit](https://xunit.net), [FluentAssertions](https://fluentassertions.com) and [Moq](https://github.com/devlooped/moq).

In this notebook, we will unit test some microservices in the Flixtube project.

- First we will examine existing unit tests for the `Metadata` microservice.
  - More importantly, the sample code demonstrates how to **mock a Repository/UnitOfWork with method calls to EFCore's DbContext/DbSet**.
  - Writing unit tests for the `History` microservice will be left as an exercise.
- Next we will examine existing unit tests for the `Gateway` microservice.
  - More importantly, the sample code demonstrates how to **mock an HTTPClientFactory and an HTTPClient with the associated HTTP calls**.
  - Writing additional unit test for the `Gateway` microservice will be left as an exercise.

---

### Examine the `Metadata` microservice

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Metadata -> Flixtube.Metadata`.
- Right-click the file `Flixtube.Metadata.csproj` and choose `Open to the side`.

The `Metadata` microservice is simply an ASP.NET Web API project, where we see that we have added NuGet packages such as `Microsoft.EntityFrameworkCore`, `Microsoft.EntityFrameworkCore.Design`, `Microsoft.EntityFrameworkCore.Tools`, `Microsoft.EntityFrameworkCore.SqlServer` and `RabbitMQ.Client`.

If you examine the folder structure for the `Metadata` microservice, you will find familiar classes when working with EntityFrameworkCore:

- The `Entities` folder defines the `Video` entity class with two properties `Id` and `Name` (right-click `Video.cs` and choose `Open to the side`).
- The `Data` folder defines the classes `ApplicationDbContext` (with a `DbSet<Video> videos`), `ApplicationDbContextFactory` and `SeedData` for (conditionally) seeding the database.
- The `Repositories` folder contains the classes `Repository`, `VideoRepository` and `UnitOfWork` along with the interfaces `IRepository`, `IVideoRepository` and `IUnitOfWork`.

> There are also a couple of classes for working with RabbitMQ:
> 
> - The `Messages` folder contains the class `VideoUploadedMessage` with the properties `Id` and `Name`.
> - The `Services` folder contains the class `RabbitMqSubscriberService` (right-click `RabbitMqSubscriberService.cs` and choose `Open to the side`).
>
> [RabbitMQ](https://www.rabbitmq.com) is an advanced message broker, on which `channels` and `exchanges` can be defined. Clients can then `publish` `messages` and/or subscribe to a `channel` or and `exchange`. `Messages` published to a `channel` or `exchange` are queued by the broker, which removes `messages` from the queue and sends them to any clients that have `subscribed` to that `channel` or `exchange`. This creates a reliable messaging system, where two clients (microservices) can be decoupled from each other, but still exchange messages with each other, where messages are guaranteed to be delivered from a publisher to subscribers (see [RabbitMQ Tutorial](https://www.rabbitmq.com/tutorials/tutorial-one-dotnet) and/or [RabbitMQ for Beginners](https://www.cloudamqp.com/blog/part1-rabbitmq-for-beginners-what-is-rabbitmq.html?gad_source=1&gclid=Cj0KCQiAs5i8BhDmARIsAGE4xHzEdTNYpf9XAbGEI7B-iYVtHbhnz4XqolMZNjeNCFSGiPoZ-YiNqY8aAsE4EALw_wcB) and/or [Chapter 5.8 in Bootstrapping Microservices](https://www.amazon.com/Bootstrapping-Microservices-Second-Kubernetes-Terraform-dp-1633438562/dp/1633438562/ref=dp_ob_title_bk)).
> 
> `RabbitMqSubscriberService` dependency injects `ILogger<RabbitMqSubscriberService>` (for logging) `IConfiguration` (for configuration data) and `IServiceScopeFactory` (for obtaining an instance of `UnitOfWork`) in its constructor. Also notice that `RabbitMqSubscriberService` inherits from `BackgroundService`, i.e. this service is meant to run in the background as long as the `Metadata` microservice is running. The two methods `StartAsync()` and `ExecuteAsync()` are overriden from `BackgroundService`:
> 
> - `StartAsync()` is called when the background service starts.
>   - It creates a connection to the RabbitMQ broker.
>   - It declares an exchange called `uploaded`, and connects an anonymous queue to it (the queue name is randomly generated by the broker for an anonymous queue).
> - `ExecuteAsync()` is called right after `StartAsync()`.
>   - It sets up an event listener `EventingBasicConsumer` on the anonymous channel connected to the `uploaded` exchange, i.e. when a `publisher` publishes a message to the `uploaded` exchange, the broker will send that message to this background service via the event listener `EventingBasicConsumer`.
>   - When a message arrives:
>     - The message is deserialized to an object of type `VideoUploadedMessage`, which is used to create an instance of the `Video` entity class.
>     - Then the dependency injected `IServiceScopeFactory` instance is used to retrieve an instance of `UnitOfWork`.
>     - Finally, the `UnitOfWork` instance is used to add the `Video` to the `Videos` table in the database.
>   - So how do the messages get published to the `uploaded` exchange?
>     - If you expand the folder `flixtube -> Flixtube.VideoUpload -> Flixtube.VideoUpload -> Services`, you will find the class `RabbitMqPublisherService` with an associated interface `IRabbitMqPublisherService`.
>     - This class is dependency injected in the `VideoUpload` microservice's `VideoUploadController`, and is used to publish a `VideoUploadedMessage` to the `uploaded` exchange when a video is uploaded.

Right-click the `Metadata` service's `Program.cs` file and choose `Open to the side`.

- At the top of the file, a check is done for required environment variables, which are then added to the `builder.Configuration` by stripping the environment variables of their `FLIXTUBE_` prefix (`MetadataController` dependency injects `IConfiguration` to access these environment variables).
- Further down the file, just before `builder.Build()`, the `RabbitMqSubscriberService` is added to the service conainer as a hosted service. `ApplicationDbContext` and `UnitOfWork` are also added to the service container (these are dependency injected into the `MetadataController` and the `RabbitMqSubscriberService`).
- After `builder.Build()`, a service container scope is created, in which the `SeedData.Initialize()` method is called. This method ensures the database is created, and seeded, if necessary.
- At the bottom of the file, the microservice is started and listens on the port number provided via an environment variable.

---

### Examine the `MetadataController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Metadata -> Flixtube.Metadata -> Controller`.
- Right-click the file `MetadataController.cs` and choose `Open to the side`.

The `MetadataController` accepts HTTP requests on the `/` route:

- `ILogger<MetadataController>` (for logging), `IConfiguration` (for reading configured environment variables) and `IUnitOfWork` (for communicating with the database) are dependency injected via the constructor.
- Notice the `/health` route which can be used by another service to check if the `Metadata` service is responding (i.e. a health check).
- The `/videos` GET route retrives and returns a list of video metadata from the database.
- The `/videos/{id}` GET route retrives and returns metadata for one specific video from the database.
- The `/video` POST route adds metadata for a video to the database.
- The `/video/{id}` DELETE route removes metadata for a specific video from the database.

What do we need to mock to unit test the `MetadataController`?

- We need to mock the services `ILogger<MetadataController>`, `IConfiguration` and `IUnitOfWork` that are dependency injected via the `MetadataController`'s constructor.
- To do this, we need to examine what methods are called via these interfaces in the methods we are unit testing (i.e. the methods for the various HTTP routes).
- For example, for the `/video` POST route, i.e. the AddVideo() method:
  - A `Video` is expected as an input parameter, and the return type is `Task<IActionResult>`.
  - The method `_logger.LogInformation()` is called.
  - The property `_unitOfWork.Videos` is read (returning an `IVideoRepository`).
  - The method `FirstOrDefaultAsync()` is called on the `IVideoRepository`.
  - The method `Add()` is called on the `IVideoRepository`.
  - The method `_unitOfWork.CompleteAsync()` is called.
  - A `BadRequest` with a string as the message can be returned.
  - A `Ok` with a `Video` as the message can be returned.

---

### Unit Testing the `MetadataController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Metadata -> Flixtube.Metadata.UnitTests`.
- Right-click the file `Flixtube.Metadata.UnitTests.csproj` and choose `Open to the side`.
- We see that this is an `xunit` project, to which we have added:
  - The `Moq` and `FluentAssertions` NuGet packages.
  - A project reference to the `Flixtube.Metadata` project.

In VSCode's explorer:

- Right-click the file `MetadataControllerTests.cs` and choose `Open to the side`.
- At the top of the file, we declare a number of private attributes, which we set in the `MetadataControllerTests` constructor.
- The constructor (in which we can define our test fixture):
  - Dependency injects `ITestOutputHelper` we can use to output debug messages to the console when running a test.
  - Sets up some common test data:
    - `_videos` which is a list of videos.
    - `_video1` which is a single video.
    - `Id` which is the Id of `_video1`.
  - Mocks `IVideoRepository` as `_mockVideoRepository` and its following methods:
    - `FindAsync()` which returns `_videos`. 
    - `FirstOrDefaultAsync()` which accepts `Id` and returns `_video1`.
    - `Add()` which accepts `_video1` and returns void. 
    - `Update()` which accepts any `Video` and returns void.
    - `Remove()` which accepts `_video1` and returns void.
  - Mocks `IUnitOfWork` and its following members:
    - `Videos` property which returns `_mockVideoRepository.Object` (i.e. an instance of the mocked `IVideoRepository`).
    - `CompleteAsync()` which returns any `int`.
  - Mocks `ILogger<MetadataController>` and its members:
    - You can mock any logger similarly to the sample code.
  - Mocks `IConfiguration`:
    - Currently no specific configuration is mocked.
    - But, if you wanted to mock e.g. environment variables, uncomment and modify the row `_mockConfig.Setup()` in the sample code.
  - Creates an instance of the Subject Under Test (SUT), i.e. the `MetadataController`, and passes in the parameters below:
    - `_mockLogger.Object` which is an instance of the mocked `ILogger<MetadataController>`.
    - `_mockConfig.Object` which is an instance of the mocked `IConfiguration`.
    - `_mockUnitOfWork.Object` which is an instance of the mocked `IUnitOfWork`.

All of the above mocking and test data could have been done in each test method, but a best practice is to define a common test fixture in the test class' constructor (if the test data and mocks are common for multiple test methods).

#### Unit Testing `GetVideos()`

- In the method `GetVideos_ForExistingVideos_ShouldReturnVideos()`:
  - We test the SUT's `GetVideos()` method, which returns a list of videos.
  - In the `Arrange` section, we set the `expected` result to `_videos` (most of the `Arrange` chores are already taken care of by the test fixture defined in the constructor).
  - In the `Act` section, we call the SUT's (`MetadataController`'s) `GetVideos()` method and store the result in a variable of type `IActionResult`.
  - In the `Assert` section:
    - We typecast the result to an `OkObjectResult` and use FluentAssertions to assert the `StatusCode` is `200`.
    - We typecast the `OkObjectResult`'s `Value` to a `List<Video>` as the `actual` result.
    - Then we verify the `expected` and `actual` list of videos is the same, with the same property values for each video.
    - We use the `ITestOutputHelper` to print some debug text to the console.
    - Finally, we verify the mocked `Videos.FindAsync()` method was called exactly once during the test.
- In the method `GetVideo_ForExistingId_ShouldReturnVideo()`:
  - We test the SUT's `GetVideo(string id)` method, which accepts a video Id and returns a single `video`.
  - This method is tested similarly to the previous test method, but instead of comparing a list of videos, a single video is compared, and during the test the mocked `Videos.FirstOrDefaultAsync()` is called instead of the mocked `Videos.FindAsync()`.
- In the method `AddVideo_ForNonExistingId_ShouldReturnNewlyCreatedVideo()`:
  - We test the SUT's `AddVideo(Video video)` method, which accepts a video and returns the newly added video as part of the response.
  - This method is tested similarly to the previous test method, but accepts a Video instead of a string as input, and during the test the mocked `Videos.Add()` and `CompleteAsync()` are called.
- In the method `DeleteVideo_ForExistingId_ShouldReturnDeletedVideo()`:
  - We test the SUT's `DeleteVideo(string)` method, which accepts a video id and returns the removed video as part of the response.
  - This method is tested similarly to the `GetVideo(string id)` method above, but during the test the mocked `Videos.Remove()` and `CompleteAsync()` are called.

#### Running the Unit Tests with `dotnet test`

If we just wanted to unit test the `Metadata` microservice, we could:

- Create a solution file (with e.g. `dotnet new sln -n Flixtube.Metadata.sln`).
- Add `Flixtube.Metadata` and `Flixtube.Metadata.UnitTests` to the solution `Flixtube.Metadata.sln`.
- Right-click the solution file `Flixtube.Metadata.sln` an choose `Open Solution`.
- Switch to the `Testing` view in VSCode and run the tests.

A simpler way is just to use the `dotnet` CLI:

- Alternative 1: Move into the folder that contains the solution file `Flixtube.Metadata.sln` and run `dotnet test`.
- Alternative 2: Move into the folder that contains the project file `Flixtube.Metadata.UnitTests.csproj` and run `dotnet test`.

Let's try the second alternative in the cell below.

- As you can see, the final row in the output is:
  - `Passed!` (all tests passed)
  - `Failed: 0` (0 tests failed)
  - `Passed: 4` (4 tests failed)
  - `Skipped: 0` (no tests were skipped)
  - `Total: 4` (a total of 4 tests were found)
  - `Duration: 1 s` (the total testing time was 1 second)
  - `Flixtube.Metadata.UnitTests.dll (net9.0)` (the test assembly run)

In [1]:
!cd ../flixtube/Flixtube.Metadata/Flixtube.Metadata.UnitTests && dotnet test

  Determining projects to restore...
  Restored c:\Users\PAGA\projects\devops\flixtube\Flixtube.Metadata\Flixtube.Metadata\Flixtube.Metadata.csproj (in 1.65 sec).
  Restored c:\Users\PAGA\projects\devops\flixtube\Flixtube.Metadata\Flixtube.Metadata.UnitTests\Flixtube.Metadata.UnitTests.csproj (in 1.76 sec).
  Flixtube.Metadata -> c:\Users\PAGA\projects\devops\flixtube\Flixtube.Metadata\Flixtube.Metadata\bin\Debug\net9.0\Flixtube.Metadata.dll
  Flixtube.Metadata.UnitTests -> c:\Users\PAGA\projects\devops\flixtube\Flixtube.Metadata\Flixtube.Metadata.UnitTests\bin\Debug\net9.0\Flixtube.Metadata.UnitTests.dll
Test run for c:\Users\PAGA\projects\devops\flixtube\Flixtube.Metadata\Flixtube.Metadata.UnitTests\bin\Debug\net9.0\Flixtube.Metadata.UnitTests.dll (.NETCoreApp,Version=v9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A total of 1 test files matched the specified pattern.

Passed!  - Failed:     0, Passed:     4, Skipped:     0, Total:     4, Duration: 1 s - 

---

### Unit Test the `HistoryController` in the `Flixtube.History` Microservice

- As an exercise, try:
- Adding an `xunit` project `Flixtube.History.UnitTests` to test the `Flixtube.History` microservice.
- Add NuGet packages `Moq` and `FluentAssertions`.
- Create a project reference to `Flixtube.History`.
- Add a class `HistoryControllerTests`, and write unit test for the `HistoryController`.
- Run `dotnet test` in the folder that contains the test project file (or the solution file if you define one).
- Note that the `Flixtube.History` microservice has a similar structure as the `Flixtube.Metadata` microservice, where both communicate with an SQL Server database and a RabbitMQ broker (although these services are never called since they are mocked).


---

### Examine the `Gateway` microservice

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Gateway -> Flixtube.Gateway`.
- Right-click the file `Flixtube.Gateway.csproj` and choose `Open to the side`.

The `Gateway` microservice is simply an ASP.NET Web API project, where we see that we haven't added any additional NuGet packages.

If you examine the folder structure for the `Gateway` microservice, you will find it has a simple structure:

- The `Models` folder defines the `Video` (with properties `Id` and `Name`) and `ViewHistory` (with properties `Id`, `VideoId` and `ViewedAt`).
- There is also a `Program.cs` file and a `GatewayController.cs` file in the `Controllers` folder.

Right-click the `Gateway` service's `Program.cs` file and choose `Open to the side`.

- At the top of the file, a check is done for required environment variables, which are then added to the `builder.Configuration` by stripping the environment variables of their `FLIXTUBE_` prefix (just as in the `Metadata` service).
- Further down the file, just before `builder.Build()`, 5 named `HTTPClients` are added to the service container (one for each of the other backend microservices, since the Gateway redirects HTTP traffic from the Web frontend to these).
- At the bottom of the file, the microservice is started and listens on the port number provided via an environment variable.

---

### Examine the `GatewayController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Gateway -> Flixtube.Gateway -> Controller`.
- Right-click the file `GatewayController.cs` and choose `Open to the side`.

The `GatewayController` accepts HTTP requests on the `/api` route:

- `ILogger<GatewayController>` (for logging), `IConfiguration` (for reading configured environment variables) and `IHttpClientFactory` (for communicating with the other backend microservices) are dependency injected via the constructor.
- Notice the `/health` route which can be used by another service to check if the `Gateway` service is responding (i.e. a health check).
- The `/api/metadata` GET route retrives and returns a list of video metadata from the Metadata microservice.
- The `/api/metadata/{id}` GET route retrives and returns metadata for one specific video from the Metadata microservice.
- The `/api/metadata` POST route adds metadata for a video by calling the Metadata microservice.
- The `/api/metadata/{id}` DELETE route removes metadata for a specific video by calling the Metadata microservice.
- The `/api/history` GET route retrives and returns a list of video view history from the History microservice.
- The `/api/video/{id}` GET route streams video from the VideoStreaming microservice for a specific video id.
- The `/api/video` POST route uploads a video by calling the VideoUpload microservice.
- The `/api/video/{id}` DELETE route removes a video fron storage by calling the VideoStorage microservice.

What do we need to mock to unit test the `GatewayController`?

- We need to mock the services `ILogger<GatewayController>`, `IConfiguration` and `IHttpClientFactory` that are dependency injected via the `GatewayController`'s constructor.
- To do this, we need to examine what methods are called via these interfaces in the methods we are unit testing (i.e. the methods for the various HTTP routes).
- For example, for the `/api/metadata` POST route, i.e. the AddMetadata() method:
  - A `Video` is expected as an input parameter, and the return type is `Task<IActionResult>`.
  - The method `_logger.LogInformation()` is called.
  - The `_httpClientFactory.CreateClient()` method is called returning an HTTPClient `client`.
  - The method `client.PostAsJsonAsync<Video>()` is called returning a `Video` or `null`.
  - A `NotFound` with no message can be returned.
  - A `Ok` with a `Video` as the message can be returned.

---

### Unit Testing the `GatewayController`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Gateway -> Flixtube.Gateway.UnitTests`.
- Right-click the file `Flixtube.Gateway.UnitTests.cs` and choose `Open to the side`.
- We see that this is an `xunit` project, to which we have added:
  - The `Moq` and `FluentAssertions` NuGet packages.
  - A project reference to the `Flixtube.Gateway` project.

In VSCode's explorer:

- Right-click the file `GatewayControllerTests.cs` and choose `Open to the side`.
- At the top of the file, we declare a number of private attributes, which we set in the `GatewayControllerTests` constructor.
- The constructor (in which we can define our test fixture):
  - Dependency injects `ITestOutputHelper` we can use to output debug messages to the console when running a test.
  - Sets up some common test data:
    - `_videos` which is a list of videos, and `_videos_json` which is an associated JSON string.
    - `_video1` which is a single video, and `_video1_json` which is an associated JSON string.
    - `Id` which is the Id of `_video1`.
  - Sets up some common `HTTPRequestMessage`s and `HTTPReponseMessage`s:
    - These will be used when defining a mock for the `HttpMessageHandler`.
  - Mocks `HttpMessageHandler` as `mockHttpMessageHandler`:
    - Various `SendAsync` method calls with the `HTTPRequestMessage`s and `HTTPReponseMessage`s above are mocked.
    - For example, the first `SendAsync()` mock is defined so that when a `getVideosRequest` message is sent via the HTTPClient, it should return a `getVideosResponse`.
  - Creates an instance `mockHttpClient` of an `HTTPClient`, passing in an instance of the mocked message handler `mockHttpMessageHandler.Object` as input.
  - Mocks `IHttpClientFactory` and its following method:
    - `CreateClient()` which accepts a string `MetadataClient` and returns the mocked `mockHttpClient`).
  - Mocks `ILogger<GatewayController>` and its members:
    - You can mock any logger similarly to the sample code.
  - Mocks `IConfiguration`:
    - Currently no specific configuration is mocked.
    - But, if you wanted to mock e.g. environment variables, uncomment and modify the row `_mockConfig.Setup()` in the sample code.
  - Creates an instance of the Subject Under Test (SUT), i.e. the `GatewayController`, and passes in the parameters below:
    - `_mockLogger.Object` which is an instance of the mocked `ILogger<GatewayController>`.
    - `_mockConfig.Object` which is an instance of the mocked `IConfiguration`.
    - `_mockHttpClientFactory.Object` which is an instance of the mocked `IHTTPClientFactory`.

All of the above mocking and test data could have been done in each test method, but a best practice is to define a common test fixture in the test class' constructor (if the test data and mocks are common for multiple test methods).

#### Unit Testing `GetMetadata()`

- In the method `GetMetadata_ForExistingMetadata_ShouldReturnMetadata()`:
  - We test the SUT's `GetMetadata()` method, which returns a list of video metadata.
  - In the `Arrange` section, we set the `expected` result to `_videos` (most of the `Arrange` chores are already taken care of by the test fixture defined in the constructor).
  - In the `Act` section, we call the SUT's (`GatewayController`'s) `GetMetadata()` method and store the result in a variable of type `IActionResult`.
  - In the `Assert` section:
    - We typecast the result to an `OkObjectResult` and use FluentAssertions to assert the `StatusCode` is `200`.
    - We typecast the `OkObjectResult`'s `Value` to a `List<Video>` as the `actual` result.
    - Then we verify the `expected` and `actual` list of videos is the same, with the same property values for each video.
    - We use the `ITestOutputHelper` to print some debug text to the console.
- In the method `GetMetadata_ForExistingId_ShouldReturnMetadata()`:
  - We test the SUT's `GetMetadata(string id)` method, which accepts a video Id and returns a single `video`.
  - This method is tested similarly to the previous test method, but instead of comparing a list of videos, a single video is compared.
- In the method `AddMetadata_ForNonExistingId_ShouldReturnNewlyCreatedMetadata()`:
  - We test the SUT's `AddMetadata(Video video)` method, which accepts a video and returns the newly added video as part of the response.
  - This method is tested similarly to the previous test method, but accepts a Video instead of a string as input.
- In the method `DeleteMetadata_ForExistingId_ShouldReturnDeletedMetadata()`:
  - We test the SUT's `DeleteMetadata(string)` method, which accepts a video id and returns an `OkResult` as part of the response.

#### Running the Unit Tests with `dotnet test`

If we just wanted to unit test the `Gateway` microservice, we could:

- Create a solution file (with e.g. `dotnet new sln -n Flixtube.Gateway.sln`).
- Add `Flixtube.Gateway` and `Flixtube.Gateway.UnitTests` to the solution `Flixtube.Gateway.sln`.
- Right-click the solution file `Flixtube.Gateway.sln` an choose `Open Solution`.
- Switch to the `Testing` view in VSCode and run the tests.

A simpler way is just to use the `dotnet` CLI:

- Alternative 1: Move into the folder that contains the solution file `Flixtube.Gateway.sln` and run `dotnet test`.
- Alternative 2: Move into the folder that contains the project file `Flixtube.Gateway.UnitTests.csproj` and run `dotnet test`.

Let's try the second alternative in the cell below.

- As you can see, the final row in the output is:
  - `Passed!` (all tests passed)
  - `Failed: 0` (0 tests failed)
  - `Passed: 4` (4 tests passed)
  - `Skipped: 0` (no tests were skipped)
  - `Total: 4` (a total of 4 tests were found)
  - `Duration: 232 ms` (the total testing time was 232 milliseconds)
  - `Flixtube.Gateway.UnitTests.dll (net9.0)` (the test assembly run)

In [2]:
!cd ../flixtube/Flixtube.Gateway/Flixtube.Gateway.UnitTests && dotnet test

  Determining projects to restore...
  Restored c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\Flixtube.Gateway.csproj (in 1.08 sec).
  Restored c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway.UnitTests\Flixtube.Gateway.UnitTests.csproj (in 1.36 sec).
c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\Controllers\GatewayController.cs(106,37): warning CS8602: Dereference of a possibly null reference. [c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\Flixtube.Gateway.csproj]
c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\Controllers\GatewayController.cs(139,36): warning CS8602: Dereference of a possibly null reference. [c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\Flixtube.Gateway.csproj]
  Flixtube.Gateway -> c:\Users\PAGA\projects\devops\flixtube\Flixtube.Gateway\Flixtube.Gateway\bin\Debug\net9.0\Flixtube.Gateway.dll
  Flixtube.Gateway.U

---

### Unit Test some of the other methods in the `GatewayController` in the `Flixtube.Gateway` Microservice

- As an exercise, try:
- Adding unit tests for methods interacting with the `Flixtube.History` microservice.
- Run `dotnet test` in the folder that contains the test project file (or the solution file if you define one).

---

### Conclusion

This completes the introduction to xUnit, Moq and FluentAssertions in VSCode, where we have unit tested the MetadataController in the Flixtube.Metadata project and the GatewayController in the Flixtube.Gateway project.

Next, we will look at Integration Testing with Playwright in VSCode:

- Open the file `integrationtesting.ipynb`.
- When the notebook opens in VSCode, click the text `Select Kernel` (top-right), and choose `Python Environments... => conda (Python 3.11) .conda/bin/python`.
- Now you can follow the instructions in the notebook.